In [1]:
import pysam
import sys
from multiprocessing import Pool

import time

In [2]:
def split_bam_strand(chrom, bam_file, out_path):
    # Open input and output BAM files
    watson_tag = ['W_C2T', 'W_G2A']
    crick_tag  = ['C_C2T', 'C_G2A']
    with pysam.AlignmentFile(bam_file, 'rb') as input_bam, \
            pysam.AlignmentFile(f'{out_path}/{chrom}.watson.bam', 'wb', template=input_bam) as output_bam1, \
            pysam.AlignmentFile(f'{out_path}/{chrom}.crick.bam', 'wb', template=input_bam) as output_bam2:

        for read in input_bam.fetch(chrom):
            # Check for tag value
            if read.has_tag('YS') and read.get_tag('YS') in watson_tag:
                output_bam1.write(read)
            elif read.has_tag('YS') and read.get_tag('YS') in crick_tag:
                output_bam2.write(read)

In [3]:
chrom_list =['chr'+str(x+1) for x in range(22)]

In [4]:
bam_file = "/home/wbguo/iproject/BSReadSim/test/data/WGBS/ERR2359938.mkdup.sorted.bam"
out_path = "/home/wbguo/iproject/BSReadSim/test/data/WGBS/split/"

In [5]:
with Pool(processes=22) as pool:
    for chrom in chrom_list:
        pool.apply_async(split_bam_strand, (chrom, bam_file, out_path,))
    pool.close()
    pool.join()